# Solutions

Generating solutions for each problem using LLM models of different levels

In [14]:
import json

def read_json(file_path : str) -> list | dict:
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
    
def write_json(data : dict, file_path : str):
    with open(file_path, "w") as f:
        json.dump(data, f)

In [15]:
PROBLEM_TEMPLATE = "Город {city}. {question}"

questions = read_json('data/questions.json')
cities = read_json('data/cities.json')
problems = [PROBLEM_TEMPLATE.format(city=c, question=q) for q in questions for c in cities]
problems

['Город Санкт-Петербург. Разработай план уплотнения городского центра.',
 'Город Кемерово. Разработай план уплотнения городского центра.',
 'Город Санкт-Петербург. Создай бренд города, основываясь на локальной идентичности.',
 'Город Кемерово. Создай бренд города, основываясь на локальной идентичности.']

In [16]:
from langchain_openai import ChatOpenAI
from fp2mp_eval._config import config

def init_llm(model : str, temperature : float = 0.5):
    return ChatOpenAI(
        model=model,
        base_url=config.base_url,
        api_key=config.api_key,
        temperature=temperature,
    )

In [17]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

MODELS = [
    'deepseek/deepseek-v4-pro',
    'deepseek/deepseek-r1'
]

results = []

for problem in tqdm(problems):

    def invoke(model : str):
        llm = init_llm(model)
        return llm.invoke(problem).content

    with ThreadPoolExecutor(len(MODELS)) as executor:
        solutions = list(executor.map(invoke, MODELS))

    for model, solution in zip(MODELS, solutions):
        results.append({
            'problem': problem,
            'model': model,
            'solution': solution
        })

100%|██████████| 4/4 [06:31<00:00, 97.94s/it] 


In [20]:
write_json(results, 'data/results.json')